# Where and when Bigfoot reports cluster

This notebook reads the course copy of the BFRO reports CSV from GitHub and exports two Vega-Lite specifications for the Jekyll project. Report counts describe this archive. They are affected by where and when people submit reports and which reports enter the dataset.

Run from this notebook's directory or from the repository root with Python 3, pandas, and Altair 5.5 installed (`%pip install pandas altair==5.5.0` in Jupyter).

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd
from IPython.display import display

data_url = (
    "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/"
    "main/bfro_reports_fall2022.csv"
)
reports = pd.read_csv(data_url, usecols=["state", "season", "classification", "number"])
reports = reports.dropna(subset=["number", "state"]).drop_duplicates(subset="number")
reports.head()

## 1. Rank states by the number of archived reports

Count unique report records by state. Select the ten highest counts, breaking ties alphabetically. This chart includes reports with an unknown season.

In [ ]:
state_counts = (
    reports.groupby("state", as_index=False)
    .size()
    .rename(columns={"size": "reports"})
    .sort_values(["reports", "state"], ascending=[False, True])
    .head(10)
)
state_order = state_counts["state"].tolist()

state_chart = (
    alt.Chart(state_counts)
    .mark_bar(color="#1D6E69", cornerRadiusEnd=3)
    .encode(
        x=alt.X("reports:Q", title="Number of reports", scale=alt.Scale(domainMin=0)),
        y=alt.Y("state:N", title=None, sort=state_order),
        tooltip=[
            alt.Tooltip("state:N", title="State"),
            alt.Tooltip("reports:Q", title="Reports", format=","),
        ],
    )
    .properties(width="container", height=270)
)
display(state_chart)

## 2. Compare seasonal report counts for the same states

Keep the four recorded seasons and count reports by state and season for the full set of classes and separately for each class. Fill absent combinations with zero so the grid stays in place when the dropdown changes. A common color scale makes counts comparable across dropdown choices.

In [ ]:
seasons = ["Spring", "Summer", "Fall", "Winter"]
known_seasons = reports[
    reports["state"].isin(state_order) & reports["season"].isin(seasons)
]
season_grid = pd.MultiIndex.from_product(
    [state_order, seasons], names=["state", "season"]
)
classes = sorted(known_seasons["classification"].dropna().unique())
class_options = ["All classes", *classes]

season_counts = pd.concat(
    [
        class_reports.groupby(["state", "season"])
        .size()
        .reindex(season_grid, fill_value=0)
        .rename("reports")
        .reset_index()
        .assign(report_class=label)
        for label, class_reports in [
            ("All classes", known_seasons),
            *list(known_seasons.groupby("classification")),
        ]
    ],
    ignore_index=True,
)

report_class = alt.param(
    name="selected_class",
    value="All classes",
    bind=alt.binding_select(options=class_options, name="Report classification: "),
)
season_chart = (
    alt.Chart(season_counts)
    .mark_rect(stroke="white", strokeWidth=1)
    .encode(
        x=alt.X("season:O", sort=seasons, title=None, axis=alt.Axis(orient="top", labelAngle=0)),
        y=alt.Y("state:N", sort=state_order, title=None),
        color=alt.Color(
            "reports:Q",
            title="Reports",
            scale=alt.Scale(scheme="blues", domain=[0, int(season_counts["reports"].max())]),
            legend=alt.Legend(orient="bottom", direction="horizontal"),
        ),
        tooltip=[
            alt.Tooltip("state:N", title="State"),
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("report_class:N", title="Classification"),
            alt.Tooltip("reports:Q", title="Reports", format=","),
        ],
    )
    .add_params(report_class)
    .transform_filter(alt.datum.report_class == report_class)
    .properties(width="container", height=270)
)
display(season_chart)

## Export the specifications

The Jekyll page reads these two JSON files. The exported specs contain only the aggregated values used by the charts.

In [ ]:
repo_root = Path.cwd()
if repo_root.name == "python_notebooks":
    repo_root = repo_root.parent

output_dir = repo_root / "assets" / "json"
output_dir.mkdir(parents=True, exist_ok=True)
state_chart.save(output_dir / "bfro-state-counts.json")
season_chart.save(output_dir / "bfro-season-counts.json")